# B2.14 · Building the pentest harness

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

Builds on **[B2.13 · Building the threat-modelling harness](https://spbreed.github.io/cyber-commons/lessons/B2.13.html)**.

| | |
|---|---|
| Open-source tooling | Metasploit, CAI |
| Open-weight models | Kimi K2.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


The pentest harness is the one where the loop can do real damage, so it is the
one where the bounds come before the capability.

Same four stages — recon, enumeration, exploitation, lateral movement — run as
an agent loop. What makes it lawful rather than an incident with a project plan
is three things, and all three are structural rather than procedural:

**Hard scope enforcement.** Not a note in the engagement letter: a check in the
request path that refuses a target outside scope before the packet leaves. Scope
is a property of the harness, not of the operator's attention.

**Destructive-action gating.** Some actions demonstrate; some destroy. The loop
may take the first class autonomously and must stop at the second, every time,
regardless of how confident it is.

**Operator handoff points.** Named places where a human decides. Not "the
operator supervises" — specific transitions, so that when the loop pauses
everybody knows whose decision it is.

The output that makes this defensible afterwards is the audit trail: every
target touched, every action taken, every refusal and why. A pentest you cannot
reconstruct is indistinguishable from an intrusion.

## 2 · Scope is a check in the path, not a line in a document

In [ ]:
import ipaddress

SCOPE = {"cidrs": ["10.20.0.0/24"], "hosts": ["app.test.corp.example"],
         "excluded": ["10.20.0.1"]}          # the gateway is always excluded

def in_scope(target):
    if target in SCOPE["excluded"]:
        return False, "explicitly excluded"
    if target in SCOPE["hosts"]:
        return True, "named host"
    try:
        ip = ipaddress.ip_address(target)
    except ValueError:
        return False, "not a named host and not an address in scope"
    for c in SCOPE["cidrs"]:
        if ip in ipaddress.ip_network(c):
            return True, f"inside {c}"
    return False, "outside every scoped range"

TARGETS = ["10.20.0.15", "app.test.corp.example", "10.20.0.1",
           "10.99.4.7", "prod.corp.example", "169.254.169.254"]
print(f"{'target':26s}{'allowed':9s}why")
for t in TARGETS:
    ok, why = in_scope(t)
    print(f"{t:26s}{'yes' if ok else 'REFUSE':9s}{why}")
allowed = [t for t in TARGETS if in_scope(t)[0]]
print(f"\n{len(allowed)} of {len(TARGETS)} targets are in scope")
assert "169.254.169.254" not in allowed and "prod.corp.example" not in allowed

## 3 · The loop, bounded by the check

In [ ]:
AUDIT = []

def act(phase, target, action, destructive=False):
    ok, why = in_scope(target)
    if not ok:
        AUDIT.append({"phase": phase, "target": target, "action": action,
                      "result": "REFUSED", "why": why})
        return "refused"
    if destructive:
        AUDIT.append({"phase": phase, "target": target, "action": action,
                      "result": "HELD", "why": "destructive - operator decision"})
        return "held"
    AUDIT.append({"phase": phase, "target": target, "action": action,
                  "result": "done", "why": why})
    return "done"

PLAN = [
 ("recon",       "10.20.0.15",            "port scan",            False),
 ("recon",       "10.99.4.7",             "port scan",            False),
 ("enumerate",   "app.test.corp.example", "enumerate endpoints",  False),
 ("exploit",     "10.20.0.15",            "sql injection proof",  False),
 ("exploit",     "10.20.0.15",            "drop table",           True),
 ("lateral",     "10.20.0.1",             "pivot via gateway",    False),
 ("lateral",     "169.254.169.254",       "read instance metadata", False),
]
for phase, target, action, destructive in PLAN:
    r = act(phase, target, action, destructive)
    print(f"   {phase:11s}{target:26s}{action:24s}-> {r}")

## 4 · Where it breaks — the operator who was supervising\n\nScope enforcement placed in the operator's attention rather than in the request path.

In [ ]:
def act_supervised(phase, target, action, operator_attention):
    """The same loop, gated by a human who is watching most of the time."""
    ok, _ = in_scope(target)
    if ok:
        return "done"
    return "refused" if operator_attention else "SENT OUT OF SCOPE"

out_of_scope = [(p, t, a) for p, t, a, _ in PLAN if not in_scope(t)[0]]
for attention in (1.0, 0.9, 0.5):
    # deterministic: the nth action is missed when n * attention has rolled over
    missed = [t for i, (p, t, a) in enumerate(out_of_scope)
              if act_supervised(p, t, a, (i + 1) * attention % 1 < attention
                                and i < len(out_of_scope) * attention) == "SENT OUT OF SCOPE"]
    print(f"operator catching {attention:.0%} of them -> "
          f"{len(out_of_scope) - len(missed)}/{len(out_of_scope)} refused")
print()
print("At 100% attention the outcome matches the structural check exactly. The")
print("difference is that the structural check does not have a bad week, and it")
print("does not get faster when the engagement is running late.")
print()
print("169.254.169.254 is the case that matters: it is one typo away from every")
print("cloud credential the harness could ever want, and it looks like a")
print("perfectly ordinary address.")

## 5 · The control — handoff points that are named, not implied

In [ ]:
HANDOFFS = {
 "destructive_action":   "operator approves each one, individually",
 "scope_expansion":      "written client approval before the range changes",
 "credential_use":       "operator supplies; the harness never stores",
 "lateral_to_new_host":  "operator confirms the host is in the engagement",
}
held = [a for a in AUDIT if a["result"] == "HELD"]
refused = [a for a in AUDIT if a["result"] == "REFUSED"]

print(f"{'transition':24s}who decides")
for k, v in sorted(HANDOFFS.items()):
    print(f"{k:24s}{v}")
print()
print(f"actions held for an operator : {len(held)}")
for h in held:
    print(f"   {h['target']:26s}{h['action']}")
print(f"actions refused outright     : {len(refused)}")
for r in refused:
    print(f"   {r['target']:26s}{r['action']:24s}({r['why']})")
assert held and refused

## 6 · Verify — a run you can reconstruct

In [ ]:
from collections import Counter
counts = Counter(a["result"] for a in AUDIT)
print(f"audit entries : {len(AUDIT)}")
for k in sorted(counts):
    print(f"   {k:9s}{counts[k]}")
print()
print(f"{'phase':11s}{'target':26s}{'action':24s}result")
for a in AUDIT:
    print(f"{a['phase']:11s}{a['target']:26s}{a['action']:24s}{a['result']}")
print()
print("Every target touched, every action taken, every refusal and its reason.")
print("This is the artefact that distinguishes an authorised engagement from an")
print("intrusion, and it has to exist before anyone asks for it.")
assert len(AUDIT) == len(PLAN) and counts["REFUSED"] >= 3

## What you just proved

Six of seven targets are classified against scope, with the cloud metadata address and the production host both refused before any request is made. The bounded loop refuses three actions, holds the destructive one for an operator, and completes the rest — and an attention-based gate is shown matching the structural check only at 100% attention. The run ends with a reconstructable audit trail.

## Your turn

Add 169.254.169.254 and its IPv6 equivalent to the exclusion list of any offensive tooling you run. It is the highest-value target in most cloud estates and the one most likely to be reached by accident.

---

**Next → [B2.15 · Choosing the model backbone](https://spbreed.github.io/cyber-commons/lessons/B2.15.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.14.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.14.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*